In [1]:
import os
from dotenv import load_dotenv
load_dotenv()
from langchain_openai import ChatOpenAI

from dotenv import load_dotenv

if os.environ['AICREDITS_API_KEY']:
    print("AICredits API Key is set.")
else:
    raise ValueError("AICredits API Key is not set.")

AICredits API Key is set.


In [2]:
from langchain_core.messages import HumanMessage, SystemMessage, AIMessage
from  langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

In [3]:
llm_openai = ChatOpenAI(model="gpt-5-mini",
                        base_url="https://api.aicredits.in/v1",
                        api_key=os.environ["AICREDITS_API_KEY"])

In [4]:
from pydantic import BaseModel
from typing import Literal

class llm_schema(BaseModel):
    movie_summary_flag: Literal["positive", "negative"]

llm_structured_output = llm_openai.with_structured_output(llm_schema)

## **CHAIN WITH Conditional Chains**

In [5]:
# TASK -1 [Prompt]

prompt_template = ChatPromptTemplate.from_messages([
    ("system", "You are a movie review evaluator"),
    ("human", "Please categorize the movie review as postive or negative : {input}")])

In [6]:
# TASK - 2 [LLM]
llm_structured_output = llm_openai.with_structured_output(llm_schema)

In [8]:
# TASK - 3 [Custom Runnable]
from langchain_core.runnables import RunnableLambda

def pydantic_json(input: llm_schema) -> str:
    return input.model_dump()['movie_summary_flag']

pydantic_json_lambda = RunnableLambda(pydantic_json)

## **Conditional Chain 1**

In [11]:
# TASK -1 [Prompt]
linkedin_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a LinkedIn post generator"),
    ("human", "Create a post for the following text for LinkedIn: {text}")])

# TASK - 2 [LLM]
llm_openai = ChatOpenAI(model="gpt-5-mini",
                        base_url="https://api.aicredits.in/v1",
                        api_key=os.environ["AICREDITS_API_KEY"])

# TASK - 3 [Str Parser]
str_parser = StrOutputParser()

chain_linked_in = linkedin_prompt | llm_openai | str_parser

## **Conditional Chain 2**

In [12]:
from langchain_core.runnables import RunnableParallel, RunnableLambda, RunnableBranch

In [13]:
def insta_chain(text: dict):
    text = text["text"]
    
    # TASK - 1 [Prompt]
    insta_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a LinkedIn post generator"),
    ("human", "Create a post for the following text for Instagram: {text}")])

    # TASK - 2 [LLM]
    llm_openai = ChatOpenAI(model="gpt-5-mini",
                            base_url="https://api.aicredits.in/v1",
                            api_key=os.environ["AICREDITS_API_KEY"])
    
    # TASK - 3 [Str Parser]
    str_parser = StrOutputParser()

    chain_insta = insta_prompt | llm_openai | str_parser

    result = chain_insta.invoke(text)
    return result
    
insta_chain_runnable = RunnableLambda(insta_chain)


## **Final Orchestration**

In [14]:
conditional_chain = RunnableBranch(
    (lambda x: "positive" in x, chain_linked_in),
    insta_chain_runnable
)
final_orchestrator = prompt_template | llm_structured_output | pydantic_json_lambda | conditional_chain

In [15]:
final_orchestrator.invoke({"input":"I loved this RRR movie"})

'Positivity isn’t just a feeling — it’s a strategy. When teams adopt a positive mindset, they solve problems faster, recover from setbacks more quickly, and create better outcomes. One simple habit I recommend: start meetings by sharing one small win or expressing gratitude — it shifts focus from obstacles to opportunities. \n\nWhat small practice helps you stay positive at work? Share your tips below. ✨\n\n#positivity #mindset #leadership #wellbeing #growth'